# DATA3010 Data Mining
## Data Preprocessing II: From Clean Data to Model-Ready Data
### STUDENT VERSION

Today is **not a repeat of Data Preprocessing I**.

Last class focused on cleaning and transforming messy data. Today we focus on a harder question:

> **How do we preprocess data without leaking future information or making the future model unreliable?**

### Today you will practice
1. Auditing before changing data.
2. Detecting invalid values with rules.
3. Exact duplicates vs conflicting duplicate IDs.
4. Target leakage.
5. Splitting before learned preprocessing.
6. Train-only imputation, encoding, and scaling.
7. Handling a future category not seen during training.
8. Auditing AI-generated preprocessing code.

### Rule for today
A notebook that runs is **not automatically correct**.

Keep asking:
- What changed?
- What did this step learn?
- Which rows did it learn from?

## 0. Setup
Run this cell first.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

RANDOM_STATE = 42
print("Setup complete.")

Setup complete.


## 1. Load today's new dataset

We are using a fictional subscription dataset. We are **not training a classifier today**.

Next week, `churned` could become the target.

Some problems were intentionally added so you can practice finding them.

In [2]:
raw = pd.DataFrame([
    ["C101",14,24,39.0,31.0,1,"Basic","North","Yes",2,546.0,None,"No"],
    ["C102",3,19,59.0,6.0,5,"Premium","South","No",18,177.0,"Too expensive","Yes"],
    ["C103",22,37,79.0,46.0,0,"Premium ","East","Y",1,1738.0,None,"No"],
    ["C104",2,np.nan,39.0,4.0,6,"basic"," south ","N",25,78.0,"Poor experience","Yes"],
    ["C105",18,52,59.0,34.0,1,"Plus","NORTH","yes",3,1062.0,None,"No"],
    ["C106",1,150,59.0,2.0,8,"Plus","East","no",30,59.0,"Moved to competitor","Yes"],
    ["C107",11,44,-5.0,28.0,2,"Basic","West","Yes",4,429.0,None,"No"],
    ["C108",7,31,39.0,17.0,3,"Basic","West","No",7,273.0,None,"No"],
    ["C108",7,31,39.0,17.0,3,"Basic","West","No",7,273.0,None,"No"],
    ["C109",5,-3,79.0,8.0,4,"Premium","East","Yes",16,395.0,"Not enough use","Yes"],
    ["C110",29,61,79.0,51.0,0,"Premium","North","Yes",1,2291.0,None,"No"],
    ["C111",4,28,np.nan,7.0,7,"Plus",None,"No",21,236.0,"Billing issue","Yes"],
    ["C112",16,35,59.0,np.nan,1,"Plus","South","Yes",2,944.0,None,"No"],
    ["C113",9,41,39.0,-2.0,5,"Basic","East","No",12,351.0,"Service quality","Yes"],
    ["C114",20,48,59.0,36.0,1,"Plus","North","Yes",2,1180.0,None,"No"],
    ["C114",20,48,79.0,36.0,1,"Premium","North","Yes",2,1580.0,None,"No"],
    ["C115",6,22,39.0,10.0,4,"Basic","South","No",15,234.0,"Too expensive","Yes"],
    ["C116",27,55,79.0,49.0,0,"Premium","West","Y",1,2133.0,None,"No"],
    ["C117",13,33,59.0,25.0,2,"Plus","East","N",4,767.0,None,"No"],
    ["C118",3,26,39.0,5.0,6,"Basic","South","No",22,117.0,"Support problems","Yes"],
    ["C119",24,np.nan,79.0,43.0,1,"Premium","North","Yes",1,1896.0,None,"No"],
    ["C120",8,39,59.0,19.0,3,"Plus","West","Yes",6,472.0,None,"No"],
], columns=[
    "customer_id","tenure_months","age","monthly_fee","usage_hours",
    "support_tickets","plan","region","auto_pay","last_payment_days_ago",
    "total_charges","cancellation_reason","churned"
])

print("Raw shape:", raw.shape)
display(raw.head(8))

Raw shape: (22, 13)


,customer_id,tenure_months,age,monthly_fee,usage_hours,support_tickets,plan,region,auto_pay,last_payment_days_ago,total_charges,cancellation_reason,churned
0,C101,14,24.0,39.0,31.0,1,Basic,North,Yes,2,546.0,NaN,No
1,C102,3,19.0,59.0,6.0,5,Premium,South,No,18,177.0,Too expensive,Yes
2,C103,22,37.0,79.0,46.0,0,Premium,East,Y,1,1738.0,NaN,No
3,C104,2,NaN,39.0,4.0,6,basic,south,N,25,78.0,Poor experience,Yes
4,C105,18,52.0,59.0,34.0,1,Plus,NORTH,yes,3,1062.0,NaN,No
5,C106,1,150.0,59.0,2.0,8,Plus,East,no,30,59.0,Moved to competitor,Yes
6,C107,11,44.0,-5.0,28.0,2,Basic,West,Yes,4,429.0,NaN,No
7,C108,7,31.0,39.0,17.0,3,Basic,West,No,7,273.0,NaN,No


## Exercise 1: Audit before changing anything

Before cleaning, inspect the evidence.

Complete each TODO.

In [3]:
# TODO 1A: Show number of rows and columns.
# Hint: raw.shape
print(raw.shape)

# TODO 1B: Show data types.
# Hint: raw.dtypes
print(raw.dtypes)

# TODO 1C: Count missing values in each column.
# Hint: raw.isna().sum()
print(raw.isna().sum())

# TODO 1D: Inspect the categories below.
# Use value_counts(dropna=False)
# Columns: plan, region, auto_pay, churned
print(raw["plan"].value_counts(dropna=False))
print(raw["region"].value_counts(dropna=False))
print(raw["auto_pay"].value_counts(dropna=False))
print(raw["churned"].value_counts(dropna=False))

# TODO 1E: Find exact duplicate rows.
# Hint: raw.duplicated(keep=False)
display(raw[raw.duplicated(keep=False)])

# TODO 1F: Find repeated customer_id values.
# Hint: raw["customer_id"].duplicated(keep=False)
display(raw[raw["customer_id"].duplicated(keep=False)])


(22, 13)
customer_id                  str
tenure_months              int64
age                      float64
monthly_fee              float64
usage_hours              float64
support_tickets            int64
plan                         str
region                       str
auto_pay                     str
last_payment_days_ago      int64
total_charges            float64
cancellation_reason          str
churned                      str
dtype: object
customer_id               0
tenure_months             0
age                       2
monthly_fee               1
usage_hours               1
support_tickets           0
plan                      0
region                    1
auto_pay                  0
last_payment_days_ago     0
total_charges             0
cancellation_reason      14
churned                   0
dtype: int64
plan
Basic       7
Plus        7
Premium     6
Premium     1
basic       1
Name: count, dtype: int64
region
North      5
East       5
West       5
South      4
 south     

,customer_id,tenure_months,age,monthly_fee,usage_hours,support_tickets,plan,region,auto_pay,last_payment_days_ago,total_charges,cancellation_reason,churned
7,C108,7,31.0,39.0,17.0,3,Basic,West,No,7,273.0,NaN,No
8,C108,7,31.0,39.0,17.0,3,Basic,West,No,7,273.0,NaN,No


,customer_id,tenure_months,age,monthly_fee,usage_hours,support_tickets,plan,region,auto_pay,last_payment_days_ago,total_charges,cancellation_reason,churned
7,C108,7,31.0,39.0,17.0,3,Basic,West,No,7,273.0,NaN,No
8,C108,7,31.0,39.0,17.0,3,Basic,West,No,7,273.0,NaN,No
14,C114,20,48.0,59.0,36.0,1,Plus,North,Yes,2,1180.0,NaN,No
15,C114,20,48.0,79.0,36.0,1,Premium,North,Yes,2,1580.0,NaN,No


**Write 2–3 sentences:** What problems did you find? Which one should not be fixed automatically?
\
I found that there are a couple duplicated rows, and multiple duplicated customer_id 's.

The duplicate rows should not be fixed automatically, there could be a reason behind its inclusion.

## 2. Validation rules

Missing and invalid are not the same thing.

Use these rules:
- age: 0 to 120
- monthly_fee: greater than 0
- usage_hours: 0 or greater
- tenure_months: 0 or greater
- support_tickets: 0 or greater

In [9]:
# TODO 2A: Create Boolean masks.
# Important: missing values are UNKNOWN, not automatically INVALID.
#
# Example:
# invalid_age = ~raw["age"].between(0, 120) & raw["age"].notna()

invalid_age = (~raw["age"].between(0, 120)) & (raw["age"].notna())
invalid_fee = (raw["monthly_fee"] > 0) & (raw["monthly_fee"].notna())
invalid_usage = (raw["usage_hours"] >= 0) & (raw["usage_hours"].notna())
invalid_tenure = (raw["tenure_months"] >= 0) & (raw["usage_hours"].notna())
invalid_tickets = (raw["support_tickets"] >= 0) & (raw["usage_hours"].notna())

# TODO 2B: Combine all masks with | into invalid_any.
invalid_any = (
    invalid_age | invalid_fee | invalid_usage | invalid_tenure | invalid_tickets
)

# TODO 2C: Display only rows that violate at least one rule.
display(raw[invalid_any])

,customer_id,tenure_months,age,monthly_fee,usage_hours,support_tickets,plan,region,auto_pay,last_payment_days_ago,total_charges,cancellation_reason,churned
0,C101,14,24.0,39.0,31.0,1,Basic,North,Yes,2,546.0,NaN,No
1,C102,3,19.0,59.0,6.0,5,Premium,South,No,18,177.0,Too expensive,Yes
2,C103,22,37.0,79.0,46.0,0,Premium,East,Y,1,1738.0,NaN,No
3,C104,2,NaN,39.0,4.0,6,basic,south,N,25,78.0,Poor experience,Yes
4,C105,18,52.0,59.0,34.0,1,Plus,NORTH,yes,3,1062.0,NaN,No
5,C106,1,150.0,59.0,2.0,8,Plus,East,no,30,59.0,Moved to competitor,Yes
6,C107,11,44.0,-5.0,28.0,2,Basic,West,Yes,4,429.0,NaN,No
7,C108,7,31.0,39.0,17.0,3,Basic,West,No,7,273.0,NaN,No
8,C108,7,31.0,39.0,17.0,3,Basic,West,No,7,273.0,NaN,No
9,C109,5,-3.0,79.0,8.0,4,Premium,East,Yes,16,395.0,Not enough use,Yes


**Question:** Why should a missing age and an age of 150 be treated differently?
One is a clerical error and the other is invalid data.

## 3. Standardize category labels

Clean the **representation**, not the meaning.

Examples:
- `Premium ` and `Premium`
- `Y`, `yes`, and `Yes`
- ` south ` and `South`

In [10]:
clean = raw.copy()

# TODO 3A: Standardize plan and region.
# Hint:
# clean["plan"] = clean["plan"].str.strip().str.title()
clean.plan = clean.plan.str.strip().str.title()
clean.region = clean.region.str.strip().str.title()


# TODO 3B: Standardize auto_pay.
# Suggested steps:
# 1. strip spaces
# 2. lowercase
# 3. map y/yes -> Yes and n/no -> No

# YOUR CODE HERE
clean.auto_pay = (
    clean.auto_pay.str.strip(" ")
    .str.lower()
    .map({"y": "Yes", "yes": "Yes", "n": "No", "no": "No"})
)

display(clean[["plan", "region", "auto_pay"]].drop_duplicates())

,plan,region,auto_pay
0,Basic,North,Yes
1,Premium,South,No
2,Premium,East,Yes
3,Basic,South,No
4,Plus,North,Yes
5,Plus,East,No
6,Basic,West,Yes
7,Basic,West,No
10,Premium,North,Yes
11,Plus,NaN,No


## 4. Exact duplicates vs conflicting duplicate IDs

An exact duplicate repeats the entire row.

A conflicting duplicate ID means the same ID appears more than once but some fields disagree.

**Do not automatically keep the first conflicting row.**

In [11]:
# TODO 4A: Display exact duplicate rows.
display(raw[raw.duplicated(keep=False)])

# TODO 4B: Display all rows with repeated customer_id values.
display(raw[raw["customer_id"].duplicated(keep=False)])

# TODO 4C: Remove ONLY exact duplicate rows.
# Save as no_exact_dupes.
no_exact_dupes = raw.drop_duplicates()
display(no_exact_dupes)

,customer_id,tenure_months,age,monthly_fee,usage_hours,support_tickets,plan,region,auto_pay,last_payment_days_ago,total_charges,cancellation_reason,churned
7,C108,7,31.0,39.0,17.0,3,Basic,West,No,7,273.0,NaN,No
8,C108,7,31.0,39.0,17.0,3,Basic,West,No,7,273.0,NaN,No


,customer_id,tenure_months,age,monthly_fee,usage_hours,support_tickets,plan,region,auto_pay,last_payment_days_ago,total_charges,cancellation_reason,churned
7,C108,7,31.0,39.0,17.0,3,Basic,West,No,7,273.0,NaN,No
8,C108,7,31.0,39.0,17.0,3,Basic,West,No,7,273.0,NaN,No
14,C114,20,48.0,59.0,36.0,1,Plus,North,Yes,2,1180.0,NaN,No
15,C114,20,48.0,79.0,36.0,1,Premium,North,Yes,2,1580.0,NaN,No


,customer_id,tenure_months,age,monthly_fee,usage_hours,support_tickets,plan,region,auto_pay,last_payment_days_ago,total_charges,cancellation_reason,churned
0,C101,14,24.0,39.0,31.0,1,Basic,North,Yes,2,546.0,NaN,No
1,C102,3,19.0,59.0,6.0,5,Premium,South,No,18,177.0,Too expensive,Yes
2,C103,22,37.0,79.0,46.0,0,Premium,East,Y,1,1738.0,NaN,No
3,C104,2,NaN,39.0,4.0,6,basic,south,N,25,78.0,Poor experience,Yes
4,C105,18,52.0,59.0,34.0,1,Plus,NORTH,yes,3,1062.0,NaN,No
5,C106,1,150.0,59.0,2.0,8,Plus,East,no,30,59.0,Moved to competitor,Yes
6,C107,11,44.0,-5.0,28.0,2,Basic,West,Yes,4,429.0,NaN,No
7,C108,7,31.0,39.0,17.0,3,Basic,West,No,7,273.0,NaN,No
9,C109,5,-3.0,79.0,8.0,4,Premium,East,Yes,16,395.0,Not enough use,Yes
10,C110,29,61.0,79.0,51.0,0,Premium,North,Yes,1,2291.0,NaN,No


**Question:** Why is this risky?

```python
clean.drop_duplicates(subset="customer_id", keep="first")
```

## 5. Target leakage

Imagine next week we want to predict `churned`.

The column `cancellation_reason` is normally known **after** the customer has churned.

Using it to predict churn would let the future model cheat.

That is **target leakage**.

In [28]:
# TODO 5A: y should contain only the target churned.

# TODO 5B: X should exclude:
# - churned
# - cancellation_reason
# - customer_id
#
# Start from no_exact_dupes.
excluded = ["churned", "cancellation_reason", "customer_id"]

y = no_exact_dupes["churned"].copy()
X = no_exact_dupes.drop(columns=excluded, errors="ignore").copy()

# TODO 5C: Print the columns that remain in X.
print(list(X.columns))


['tenure_months', 'age', 'monthly_fee', 'usage_hours', 'support_tickets', 'plan', 'region', 'auto_pay', 'last_payment_days_ago', 'total_charges']


### Leakage check

Decide Keep or Exclude and explain why:

| Column | Keep / Exclude | Why? |
|---|---|---|
| tenure_months | Keep | Would long term customers be less lilely to churn? |
| support_tickets | Keep | Would a spike in tickets indicate a higher risk or churning? |
| cancellation_reason | Exclude | This data is only generated after the user churns |
| customer_id | Exclude | It is arbitrary and is not a predictive signal |
| monthly_fee | Keep | It represents pricing sensitivity at a tier level |

**Challenge:** Can a feature be extremely accurate and still be a bad feature?
There are features that are generated after production, and would not be helpful to predict in the moment

## 6. Split before learned preprocessing

Some preprocessing steps learn from data:
- median imputation
- most-frequent imputation
- standardization
- categories for one-hot encoding

Safe order:

**Split → fit preprocessing on training data → transform train/test**

In [32]:
# TODO 6: Split X and y.
#
# Use:
# test_size=0.30
# random_state=RANDOM_STATE
# stratify=y

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

**Question:** Why use `stratify=y`?
stratify=y ensures that random sampling doesn't provide a test set too few. 

## 7. Build the preprocessing pipeline

Numeric columns:
- median imputation
- standardization

Categorical columns:
- most-frequent imputation
- one-hot encoding
- ignore unseen future categories

Focus on the logic. You do not need to memorize all the syntax.

In [33]:
numeric_features = [
    "tenure_months",
    "age",
    "monthly_fee",
    "usage_hours",
    "support_tickets",
    "last_payment_days_ago",
    "total_charges",
]

categorical_features = ["plan", "region", "auto_pay"]

# TODO 7A: Build numeric_pipe using:
# SimpleImputer(strategy="median")
# StandardScaler()
numeric_pipe = Pipeline(
    [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

# TODO 7B: Build categorical_pipe using:
# SimpleImputer(strategy="most_frequent")
# OneHotEncoder(handle_unknown="ignore", sparse_output=False)
#
# If sparse_output=False gives an error on an older computer,
# use sparse=False instead.
categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

# TODO 7C: Combine both with ColumnTransformer.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features),
    ]
)

## 8. Fit on training data only

Training data:
```python
fit_transform(...)
```

Test/future data:
```python
transform(...)
```

Do not fit again on test or future data.

In [34]:
# TODO 8A:
# X_train_ready = preprocessor.fit_transform(X_train)

# TODO 8B:
# X_test_ready = preprocessor.transform(X_test)

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

**Explain:** Why is this unsafe?

```python
all_ready = preprocessor.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(all_ready, y)
```

## 9. Inspect what preprocessing learned

After fitting, inspect the learned medians, scaler statistics, and categories.

In [ ]:
# TODO 9:
# numeric_imputer = preprocessor.named_transformers_["num"].named_steps["imputer"]
# scaler = preprocessor.named_transformers_["num"].named_steps["scaler"]
# encoder = preprocessor.named_transformers_["cat"].named_steps["encoder"]
#
# Print:
# numeric_imputer.statistics_
# scaler.mean_
# encoder.categories_

# YOUR CODE HERE

## 10. Future unseen category

A new plan called `Student` did not exist in training.

We want the existing preprocessing to handle the row without learning from it.

In [ ]:
future_customer = pd.DataFrame([{
    "tenure_months": 2,
    "age": 21,
    "monthly_fee": 29.0,
    "usage_hours": 12.0,
    "support_tickets": 1,
    "plan": "Student",
    "region": "West",
    "auto_pay": "Yes",
    "last_payment_days_ago": 2,
    "total_charges": 58.0
}])

# TODO 10:
# Use preprocessor.transform(future_customer)
# Do NOT call fit_transform().
future_ready = None

**Question:** Why would fitting again on the future customer be wrong?

## 11. AI audit

**Only use AI here if your instructor explicitly allows it for this activity.**

Do not paste course datasets, notebooks, another student's work, or private information into an AI system.

You may ask a generic question such as:

> What is data leakage in preprocessing, and how can scaling or imputation cause it?

Then audit this hypothetical AI-generated code:

```python
df = raw.dropna()
df = df.drop_duplicates(subset="customer_id", keep="first")

X = df.drop(columns=["churned"])
y = df["churned"]

X = pd.get_dummies(X)
X = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y)
```

### Your task

Find **at least four problems or risks**.

For each:
1. What is wrong?
2. Why does it matter?
3. How would you improve it?

## Optional challenge for students who have taken Machine Learning

1. Why will scaling matter much more for KNN than for a basic decision tree?
2. Why is `customer_id` usually a bad predictive feature?
3. Why should an unseen category in future data **not** cause us to refit the encoder?

# Exit ticket

1. What is target leakage?
2. Why split before fitting an imputer or scaler?
3. What is the difference between an exact duplicate and a conflicting duplicate ID?
4. If preprocessing code runs successfully, does that prove it is correct? Why?

**Bridge to Wednesday:**  
Today we made data model-ready without cheating. Next class we begin classification.